# Encoder Analysis

1. Checking collapse
2. Confounder probe

In [ ]:
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

from src.config import DiffusionConfig, ModelConfig
from src.data import load_ihdp, make_ihdp_confounded
from src.model import HybridModel

In [10]:
MODEL_CFG = ModelConfig(feature_dim=25, latent_dim=20, hidden_dim=64, num_layers=2)
DIFF_CFG = DiffusionConfig(
    num_steps=100,
    beta_start=0.0001,
    beta_end=0.2,
    schedule="quad",
    embedding_dim=32,
    block_dim=32,
    hidden_dim=32,
    num_blocks=4,
    clip_denoised=True,
)
CKPT_PATH = "checkpoints/final_model_hybrid_conf_2026-08-04T12_59_25.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
train_ds, val_ds, test_ds, y_std = load_ihdp(
    "data/ihdp", replication=1, train_ratio=0.7, test_ratio=0.15
)
train_ds, val_ds, test_ds = (make_ihdp_confounded(ds) for ds in (train_ds, val_ds, test_ds))

model = HybridModel(MODEL_CFG, DIFF_CFG).to(device)
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model.eval()


def get_mu_and_sigma(ds) -> tuple[torch.Tensor, torch.Tensor]:
    x, a, y = ds.x.to(device), ds.a.to(device), ds.y.to(device)
    with torch.no_grad():
        _, mu, sigma = model.encoder.rsample(x, a, y)
    return mu.cpu(), sigma.cpu()

## Checking Collapse

This section is for checking our model's latent `z` for posterior collapse on a `hybrid_*` checkpoint.

If the encoder has collapsed to the prior N(0,I), `mu` will sit near 0 and `sigma` near 1 for most latent dimensions, with little variation across subjects -- meaning `z` carries almost no information about `(x, a, y_fac)`, regardless of `latent_dim`/`hidden_dim` size.

In [ ]:
train_ds, val_ds, test_ds, y_std = load_ihdp(
    "data/ihdp", replication=1, train_ratio=0.7, test_ratio=0.15
)
train_ds, val_ds, test_ds = (
    make_ihdp_confounded(ds, effect=0.4) for ds in (train_ds, val_ds, test_ds)
)

model = HybridModel(MODEL_CFG, DIFF_CFG).to(device)
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model.eval()

mu, sigma = get_mu_and_sigma(train_ds)
print(f"z shape: {mu.shape}  (N={mu.shape[0]}, latent_dim={mu.shape[1]})")

z shape: torch.Size([689, 20])  (N=689, latent_dim=20)


In [13]:
print("Prior is N(0, 1) per dimension. Collapse looks like mu~0, sigma~1, low mu variance.\n")

print(f"{'dim':>4} {'mean|mu|':>10} {'std(mu)':>10} {'mean sigma':>11}")
for d in range(mu.shape[1]):
    print(
        f"{d:>4}"
        f" {mu[:, d].abs().mean().item():>10.4f}"
        f" {mu[:, d].std().item():>10.4f}"
        f" {sigma[:, d].mean().item():>11.4f}"
    )

print()
print(
    f"Aggregate: mean|mu|={mu.abs().mean().item():.4f}  "
    f"mean std(mu) across dims={mu.std(dim=0).mean().item():.4f}  "
    f"mean sigma={sigma.mean().item():.4f}"
)
print(
    f"KL from prior (mean over dims and subjects): "
    f"{(0.5 * (mu.pow(2) + sigma.pow(2) - 2 * sigma.log() - 1)).mean().item():.4f}"
)

Prior is N(0, 1) per dimension. Collapse looks like mu~0, sigma~1, low mu variance.

 dim   mean|mu|    std(mu)  mean sigma
   0     0.4567     0.5112      0.7777
   1     0.6241     0.7558      0.6894
   2     0.3933     0.4718      0.8349
   3     0.4970     0.4782      0.7961
   4     0.4167     0.5121      0.8273
   5     0.2865     0.3600      0.8848
   6     0.2462     0.2988      0.8769
   7     0.3421     0.3687      0.8692
   8     0.4156     0.5237      0.7902
   9     0.4057     0.4770      0.8365
  10     0.6058     0.7116      0.7482
  11     0.3580     0.4330      0.8512
  12     0.7269     0.8240      0.6307
  13     0.4974     0.5796      0.7629
  14     0.4241     0.5047      0.8085
  15     0.4280     0.4874      0.8163
  16     0.3779     0.4564      0.8511
  17     0.4699     0.5131      0.8015
  18     0.6309     0.7515      0.7292
  19     0.3956     0.4824      0.8310

Aggregate: mean|mu|=0.4499  mean std(mu) across dims=0.5251  mean sigma=0.8007
KL from prior (m

## Confounder Probe

This section probes *'does z actually encode the hidden confounder?'*

Trains a logistic regression on `z` (the trained encoder's posterior mean) to predict `momblack`, and compares against the same probe trained directly on `x` -- the raw covariates `z` was derived from. If `x` predicts `momblack` better than `z` does, the encoder is losing confounder-relevant signal during encoding, not just failing to have any (posterior collapse, which has been ruled out above).

In [14]:
x_train, x_test = train_ds.x.numpy(), test_ds.x.numpy()
z_train, _ = get_mu_and_sigma(train_ds)
z_test, _ = get_mu_and_sigma(test_ds)
conf_train = train_ds.confounder.astype(int)
conf_test = test_ds.confounder.astype(int)

In [15]:
base_rate = conf_test.mean()
majority_acc = max(base_rate, 1 - base_rate)
print(f"N train={len(conf_train)}  N test={len(conf_test)}  test base rate={base_rate:.4f}")
print(f"Majority-class baseline accuracy: {majority_acc:.4f}\n")

N train=689  N test=148  test base rate=0.5068
Majority-class baseline accuracy: 0.5068



In [16]:
print(f"{'probe input':<12} {'accuracy':>10} {'AUC':>10}")
for name, xtr, xte in (
    ("x (25-dim)", x_train, x_test),
    ("z (20-dim)", z_train.numpy(), z_test.numpy()),
):
    clf = LogisticRegression(max_iter=2000).fit(xtr, conf_train)
    pred = clf.predict(xte)
    proba = clf.predict_proba(xte)[:, 1]
    acc = accuracy_score(conf_test, pred)
    auc = roc_auc_score(conf_test, proba)
    print(f"{name:<12} {acc:>10.4f} {auc:>10.4f}")

probe input    accuracy        AUC
x (25-dim)       0.7297     0.8405
z (20-dim)       0.7230     0.8142
